# Qwen3-14B + LoRA on Trainium2 -- Batch Size Testing

This notebook tests **batch_size=4** for Qwen3-14B with LoRA on trn2.3xlarge.
It compares throughput against the bs=1 baseline (55.3 tok/s from the bs=1 notebook).

## Approach

The text-only `NeuronQwen3ForCausalLM` uses `ModelWrapper._forward_with_pad()` which
supports batch_size > 1 natively in NxDI SDK 2.28 -- no patches required.
(The Chandra 3-bug patch is only needed for VL models, not text-only.)

## Requirements

- **Instance**: trn2.3xlarge (4 NeuronCores at LNC=2)
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260227
- **SDK**: Neuron SDK 2.28, NxDI 0.8.x, vLLM-neuron 0.4.1
- **TP**: 4, **Batch size**: 4
- **Precision**: BF16, ISA kernels: QKV+Attn ON, MLP OFF

## 0. Environment Setup

In [ ]:
import os
import sys
import time
import json

os.environ["VLLM_NEURON_FRAMEWORK"] = "neuronx-distributed-inference"
os.environ["VLLM_USE_V1"] = "1"

# Configuration
BASE_MODEL_ID = "Qwen/Qwen3-14B"
BASE_MODEL_DIR = os.path.expanduser("~/Qwen3-14B")
LORA_DIR = os.path.expanduser("~/lora_adapters/qwen3-14b")
ARTIFACT_DIR = os.path.expanduser("~/neuron-artifacts/qwen3-14b-tp4-bs4-lora")
HF_TOKEN = os.getenv("HF_TOKEN")  # Set your HF token as an env var

LORA_ADAPTERS = {
    "adapter_a": "nicoboss/Qwen3-14B-Uncensored-Lora",
    "adapter_b": "Wuhall/Qwen3-14B-LoRA",
}

TP_DEGREE = 4
MAX_MODEL_LEN = 4096
BATCH_SIZE = 4  # Testing batch_size=4 (was 1 in previous notebook)

print(f"Base model: {BASE_MODEL_ID}")
print(f"TP degree: {TP_DEGREE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"LoRA adapters: {list(LORA_ADAPTERS.keys())}")

Base model: Qwen/Qwen3-14B
TP degree: 4
Batch size: 4
Artifacts: /home/ubuntu/neuron-artifacts/qwen3-14b-tp4-bs4-lora
LoRA adapters: ['adapter_a', 'adapter_b']


In [2]:
!pip install -q huggingface_hub safetensors

## 1. Download Model and LoRA Adapters

In [3]:
from huggingface_hub import snapshot_download, hf_hub_download

if not os.path.exists(os.path.join(BASE_MODEL_DIR, "config.json")):
    print(f"Downloading {BASE_MODEL_ID}...")
    snapshot_download(
        repo_id=BASE_MODEL_ID,
        local_dir=BASE_MODEL_DIR,
        ignore_patterns=["*.gguf", "*.md", "original/*"],
        token=HF_TOKEN,
    )
    print(f"Downloaded to {BASE_MODEL_DIR}")
else:
    print(f"Base model already exists at {BASE_MODEL_DIR}")

Downloaded to /home/ubuntu/Qwen3-14B



Fetching 17 files: 100%|██████████| 17/17 [01:03<00:00,  3.72s/it]


In [4]:
with open(os.path.join(BASE_MODEL_DIR, "config.json")) as f:
    config = json.load(f)

print(f"Model type: {config['model_type']}")
print(f"Hidden size: {config['hidden_size']}")
print(f"Num layers: {config['num_hidden_layers']}")
print(f"tie_word_embeddings: {config.get('tie_word_embeddings', 'not set')}")

assert config.get("tie_word_embeddings", True) == False, \
    "Expected tie_word_embeddings=False for Qwen3-14B"

Model type: qwen3
Hidden size: 5120
Num layers: 40
tie_word_embeddings: False


In [5]:
os.makedirs(LORA_DIR, exist_ok=True)

for adapter_id, repo_id in LORA_ADAPTERS.items():
    adapter_path = os.path.join(LORA_DIR, adapter_id)
    if os.path.exists(os.path.join(adapter_path, "adapter_config.json")):
        print(f"  {adapter_id} already downloaded")
        continue
    
    print(f"  Downloading {repo_id} -> {adapter_path}")
    os.makedirs(adapter_path, exist_ok=True)
    
    hf_hub_download(repo_id=repo_id, filename="adapter_config.json",
                    local_dir=adapter_path, token=HF_TOKEN)
    hf_hub_download(repo_id=repo_id, filename="adapter_model.safetensors",
                    local_dir=adapter_path, token=HF_TOKEN)
    print(f"  Downloaded {adapter_id}")

for adapter_id in LORA_ADAPTERS:
    adapter_path = os.path.join(LORA_DIR, adapter_id)
    with open(os.path.join(adapter_path, "adapter_config.json")) as f:
        acfg = json.load(f)
    print(f"\n{adapter_id}: rank={acfg.get('r')}, alpha={acfg.get('lora_alpha')}")

  Downloaded adapter_a
  Downloaded adapter_b

adapter_a: rank=32, alpha=16

adapter_b: rank=32, alpha=32


## 2. Create LoRA JSON Config

In [6]:
lora_config = {
    "lora-ckpt-dir": LORA_DIR,
    "lora-ckpt-paths": {
        adapter_id: adapter_id
        for adapter_id in LORA_ADAPTERS
    },
    "lora-ckpt-paths-cpu": {}
}

lora_json_path = os.path.join(LORA_DIR, "adapters.json")
with open(lora_json_path, "w") as f:
    json.dump(lora_config, f, indent=2)

print(f"LoRA config written to: {lora_json_path}")
print(json.dumps(lora_config, indent=2))

LoRA config written to: /home/ubuntu/lora_adapters/qwen3-14b/adapters.json
{
  "lora-ckpt-dir": "/home/ubuntu/lora_adapters/qwen3-14b",
  "lora-ckpt-paths": {
    "adapter_a": "adapter_a",
    "adapter_b": "adapter_b"
  },
  "lora-ckpt-paths-cpu": {}
}


## 3. Compile Model at batch_size=4

Key difference from bs=1 notebook:
- `batch_size`, `ctx_batch_size`, `tkg_batch_size` = 4
- `max_num_seqs` = 4

This compiles separate NEFFs for batch_size=4, allowing 4 requests to be processed
simultaneously in a single forward pass.

In [7]:
text_neuron_config = {
    "batch_size": BATCH_SIZE,
    "ctx_batch_size": BATCH_SIZE,
    "tkg_batch_size": BATCH_SIZE,
    "seq_len": MAX_MODEL_LEN,
    "max_context_length": MAX_MODEL_LEN,
    "torch_dtype": "bfloat16",
    "tp_degree": TP_DEGREE,
    "world_size": TP_DEGREE,
    "enable_bucketing": True,
    "context_encoding_buckets": [512, 1024, 2048, 4096],
    "token_generation_buckets": [512, 1024, 2048, 4096],
    "fused_qkv": True,
    "qkv_kernel_enabled": True,
    "mlp_kernel_enabled": False,
    "attn_kernel_enabled": True,
    "logical_neuron_cores": 2,
    "cc_pipeline_tiling_factor": 2,
    "rpl_reduce_dtype": "bfloat16",
    "attention_dtype": "bfloat16",
    "cast_type": "as-declared",
    "save_sharded_checkpoint": True,
}

print("Text neuron config (batch_size=4):")
for k, v in text_neuron_config.items():
    print(f"  {k}: {v}")

Text neuron config (batch_size=4):
  batch_size: 4
  ctx_batch_size: 4
  tkg_batch_size: 4
  seq_len: 4096
  max_context_length: 4096
  torch_dtype: bfloat16
  tp_degree: 4
  world_size: 4
  enable_bucketing: True
  context_encoding_buckets: [512, 1024, 2048, 4096]
  token_generation_buckets: [512, 1024, 2048, 4096]
  fused_qkv: True
  qkv_kernel_enabled: True
  mlp_kernel_enabled: False
  attn_kernel_enabled: True
  logical_neuron_cores: 2
  cc_pipeline_tiling_factor: 2
  rpl_reduce_dtype: bfloat16
  attention_dtype: bfloat16
  cast_type: as-declared
  save_sharded_checkpoint: True


In [8]:
os.environ["NEURON_COMPILED_ARTIFACTS"] = ARTIFACT_DIR
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print(f"Artifacts will be saved to: {ARTIFACT_DIR}")
print(f"\nStarting compilation (batch_size={BATCH_SIZE})...")
print(f"This will take 10-30 minutes on trn2.3xlarge.")

from vllm import LLM, SamplingParams

compile_start = time.time()

llm = LLM(
    model=BASE_MODEL_DIR,
    tokenizer=BASE_MODEL_DIR,
    trust_remote_code=True,
    dtype="bfloat16",
    tensor_parallel_size=TP_DEGREE,
    max_num_seqs=BATCH_SIZE,
    max_model_len=MAX_MODEL_LEN,
    swap_space=0,
    enable_lora=True,
    max_loras=len(LORA_ADAPTERS),
    max_lora_rank=32,
    additional_config=dict(
        override_neuron_config=dict(
            text_neuron_config=text_neuron_config,
            lora_ckpt_json=lora_json_path,
        )
    ),
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)

compile_time = time.time() - compile_start
print(f"\nCompilation + loading completed in {compile_time:.1f}s ({compile_time/60:.1f} min)")

Artifacts will be saved to: /home/ubuntu/neuron-artifacts/qwen3-14b-tp4-bs4-lora

Starting compilation (batch_size=4)...
This will take 10-30 minutes on trn2.3xlarge.
INFO 03-18 14:38:08 [__init__.py:43] Available plugins for group vllm.platform_plugins:
INFO 03-18 14:38:08 [__init__.py:45] - neuron -> vllm_neuron:register
INFO 03-18 14:38:08 [__init__.py:48] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 03-18 14:38:09 [__init__.py:217] Platform plugin neuron is activated
INFO 03-18 14:38:48 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 03-18 14:38:48 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
INFO 03-18 14:39:01 [utils.py:253] non-default args: {'tokenizer': '/home/ubuntu/Qwen3-14B', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 4096, 'tensor_parallel_size': 4, 'e

[2026-03-18 14:39:01] INFO platform.py:108: Applying Neuron config overrides
[2026-03-18 14:39:01] INFO platform.py:125: Neuron config overrides applied successfully
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
[2026-03-18 14:39:38] INFO platform_overrides.py:22: Skipping attention head divisibility check for Neuron platform
[2026-03-18 14:39:38] INFO platform.py:167: Neuron OpenAI serving overrides applied successfully
[2026-03-18 14:39:38] INFO platform.py:272: The custom Neuron scheduler will disable chunked prefill and schedule requests using the continuous batching mechanism, prioritizing prefill over decode.
[2026-03-18 14:39:38] INFO platform.py:285: Neuron custom scheduler default: max_num_batched_tokens set to 131072. Override with --max-num-batched-tokens if needed.
[2026-03-18 14:39:38] WARNING platform.py:312: Pin memory is not supported on Neuron.


## 4. Verify Inference

Quick smoke test with single and batch requests.

In [9]:
from vllm.lora.request import LoRARequest

sampling = SamplingParams(top_k=1, max_tokens=128, temperature=0.0)

test_prompts = [
    "Explain quantum computing in one paragraph.",
    "Write a short Python function that checks if a number is prime.",
    "What are the main differences between TCP and UDP?",
    "Describe the process of photosynthesis briefly.",
]

# Single request test
print("=" * 60)
print("SINGLE REQUEST TEST (1 prompt)")
print("=" * 60)

lora_req_a = LoRARequest("adapter_a", lora_int_id=1, lora_path=" ")

t0 = time.time()
outputs = llm.generate([{"prompt": test_prompts[0]}], sampling, lora_request=[lora_req_a])
latency = time.time() - t0
n_tokens = len(outputs[0].outputs[0].token_ids)
print(f"Tokens: {n_tokens}, Latency: {latency:.2f}s, Throughput: {n_tokens/latency:.1f} tok/s")
print(f"Output: {outputs[0].outputs[0].text[:200]}")
print()

SINGLE REQUEST TEST (1 prompt)
Tokens: 128, Latency: 5.20s, Throughput: 24.6 tok/s
Output:  Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. Unlike classical computers, which use bits as t




Adding requests: 100%|██████████| 1/1 [00:00<00:00, 105.24it/s]

Processed prompts: 100%|██████████| 1/1 [00:05<00:00,  5.19s/it, est. speed input: 1.54 toks/s, output: 24.65 toks/s]


In [10]:
# Batch request test (4 prompts simultaneously)
print("=" * 60)
print(f"BATCH REQUEST TEST ({BATCH_SIZE} prompts simultaneously)")
print("=" * 60)

batch_inputs = [{"prompt": p} for p in test_prompts[:BATCH_SIZE]]
lora_reqs = [lora_req_a] * BATCH_SIZE

t0 = time.time()
outputs = llm.generate(batch_inputs, sampling, lora_request=lora_reqs)
latency = time.time() - t0

total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
print(f"Total tokens: {total_tokens}, Latency: {latency:.2f}s")
print(f"Aggregate throughput: {total_tokens/latency:.1f} tok/s")
print(f"Per-request throughput: {total_tokens/latency/BATCH_SIZE:.1f} tok/s")
print()
for i, o in enumerate(outputs):
    n = len(o.outputs[0].token_ids)
    print(f"  Prompt {i}: {n} tokens -- {o.outputs[0].text[:100]}...")

BATCH REQUEST TEST (4 prompts simultaneously)
Total tokens: 512, Latency: 9.14s
Aggregate throughput: 56.0 tok/s
Per-request throughput: 14.0 tok/s

  Prompt 0: 128 tokens --  Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposit...
  Prompt 1: 128 tokens --  The function should return True if the number is prime, and False otherwise. The function should ha...
  Prompt 2: 128 tokens --  Also, could you explain what a socket is in the context of network programming? Additionally, I'm c...
  Prompt 3: 128 tokens --  Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy ...



Adding requests: 100%|██████████| 4/4 [00:00<00:00, 2728.00it/s]

Processed prompts: 100%|██████████| 4/4 [00:09<00:00,  2.28s/it, est. speed input: 4.27 toks/s, output: 56.03 toks/s]


## 5. Benchmark: bs=4 Throughput

Structured benchmark comparing single-request and batched throughput.
We send 4 prompts at a time to fully utilize the compiled batch size.

In [11]:
import statistics

benchmark_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a recursive Fibonacci function in Python with memoization.",
    "What is the difference between a stack and a queue?",
    "Describe the process of photosynthesis.",
    "What are the advantages of microservices architecture?",
    "Explain how a neural network learns through backpropagation.",
    "What is the CAP theorem in distributed systems?",
    "Write a SQL query to find the second highest salary.",
    "Explain the concept of polymorphism in object-oriented programming.",
    "What are the key features of the Rust programming language?",
    "How does garbage collection work in Java?",
    "Explain the difference between REST and GraphQL.",
]

sampling_bench = SamplingParams(top_k=1, max_tokens=256, temperature=0.0)

print(f"Benchmark prompts: {len(benchmark_prompts)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches: {len(benchmark_prompts) // BATCH_SIZE}")

Benchmark prompts: 12
Batch size: 4
Batches: 3


In [12]:
# Benchmark: Single-request throughput (bs=4 compiled, sending 1 at a time)
print("=" * 60)
print("BENCHMARK: Single-request (adapter_a)")
print("=" * 60)

lora_req_a = LoRARequest("adapter_a", lora_int_id=1, lora_path=" ")

# Warmup
print("  Warmup...")
llm.generate([{"prompt": benchmark_prompts[0]}], sampling_bench, lora_request=[lora_req_a])

single_results = []
print(f"  Running {len(benchmark_prompts)} single prompts...")
for i, prompt in enumerate(benchmark_prompts):
    t0 = time.time()
    outputs = llm.generate([{"prompt": prompt}], sampling_bench, lora_request=[lora_req_a])
    latency = time.time() - t0
    n_tokens = len(outputs[0].outputs[0].token_ids)
    tok_per_s = n_tokens / latency if latency > 0 else 0
    single_results.append({"n_tokens": n_tokens, "latency_s": round(latency, 3), "tok_per_s": round(tok_per_s, 1)})

single_throughputs = [r["tok_per_s"] for r in single_results]
single_latencies = [r["latency_s"] for r in single_results]
print(f"\n  Single-request results:")
print(f"    Mean throughput: {statistics.mean(single_throughputs):.1f} +/- {statistics.stdev(single_throughputs):.1f} tok/s")
print(f"    Mean latency:   {statistics.mean(single_latencies):.2f}s")

BENCHMARK: Single-request (adapter_a)
  Warmup...
  Running 12 single prompts...

  Single-request results:
    Mean throughput: 33.7 +/- 0.3 tok/s
    Mean latency:   7.60s



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1387.92it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.57s/it, est. speed input: 1.45 toks/s, output: 33.81 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2034.10it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.68s/it, est. speed input: 1.43 toks/s, output: 33.32 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1271.00it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.65s/it, est. speed input: 1.44 toks/s, output: 33.48 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1468.08it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.52s/it, est. speed input: 1.46 toks/s, output: 34.07 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 2207.53it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.55s/it, est. speed input: 0.93 toks/s, output: 33.89 toks/s]

Adding requests: 100%|██████████| 1/1 [00:00<00:00, 1394.38it/s]

Processe

In [13]:
# Benchmark: Batched throughput (bs=4 compiled, sending 4 at a time)
print("=" * 60)
print(f"BENCHMARK: Batched (bs={BATCH_SIZE}, adapter_a)")
print("=" * 60)

# Create batches of BATCH_SIZE prompts
n_batches = len(benchmark_prompts) // BATCH_SIZE
batches = [benchmark_prompts[i*BATCH_SIZE:(i+1)*BATCH_SIZE] for i in range(n_batches)]

# Warmup with a full batch
print(f"  Warmup (batch of {BATCH_SIZE})...")
warmup_inputs = [{"prompt": p} for p in batches[0]]
warmup_loras = [lora_req_a] * BATCH_SIZE
llm.generate(warmup_inputs, sampling_bench, lora_request=warmup_loras)

batch_results = []
print(f"  Running {n_batches} batches of {BATCH_SIZE}...")
for batch_idx, batch_prompts in enumerate(batches):
    batch_inputs = [{"prompt": p} for p in batch_prompts]
    batch_loras = [lora_req_a] * len(batch_prompts)
    
    t0 = time.time()
    outputs = llm.generate(batch_inputs, sampling_bench, lora_request=batch_loras)
    latency = time.time() - t0
    
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    aggregate_tok_s = total_tokens / latency if latency > 0 else 0
    per_request_tok_s = aggregate_tok_s / len(batch_prompts)
    
    batch_results.append({
        "batch_idx": batch_idx,
        "num_prompts": len(batch_prompts),
        "total_tokens": total_tokens,
        "latency_s": round(latency, 3),
        "aggregate_tok_s": round(aggregate_tok_s, 1),
        "per_request_tok_s": round(per_request_tok_s, 1),
    })
    print(f"    Batch {batch_idx}: {total_tokens} tokens, {latency:.2f}s, "
          f"{aggregate_tok_s:.1f} agg tok/s, {per_request_tok_s:.1f} per-req tok/s")

batch_agg_throughputs = [r["aggregate_tok_s"] for r in batch_results]
batch_per_req_throughputs = [r["per_request_tok_s"] for r in batch_results]
batch_latencies = [r["latency_s"] for r in batch_results]

print(f"\n  Batched results (bs={BATCH_SIZE}):")
print(f"    Mean aggregate throughput: {statistics.mean(batch_agg_throughputs):.1f} +/- {statistics.stdev(batch_agg_throughputs):.1f} tok/s")
print(f"    Mean per-request throughput: {statistics.mean(batch_per_req_throughputs):.1f} tok/s")
print(f"    Mean latency (wall clock per batch): {statistics.mean(batch_latencies):.2f}s")

BENCHMARK: Batched (bs=4, adapter_a)
  Warmup (batch of 4)...
  Running 3 batches of 4...
    Batch 0: 1024 tokens, 12.32s, 83.1 agg tok/s, 20.8 per-req tok/s
    Batch 1: 1024 tokens, 12.33s, 83.0 agg tok/s, 20.8 per-req tok/s
    Batch 2: 1024 tokens, 12.15s, 84.3 agg tok/s, 21.1 per-req tok/s

  Batched results (bs=4):
    Mean aggregate throughput: 83.5 +/- 0.7 tok/s
    Mean per-request throughput: 20.9 tok/s
    Mean latency (wall clock per batch): 12.27s



Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3929.09it/s]

Processed prompts: 100%|██████████| 4/4 [00:11<00:00,  2.99s/it, est. speed input: 3.34 toks/s, output: 85.51 toks/s]

Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3806.08it/s]

Processed prompts: 100%|██████████| 4/4 [00:12<00:00,  3.08s/it, est. speed input: 3.25 toks/s, output: 83.12 toks/s]

Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3544.73it/s]

Processed prompts: 100%|██████████| 4/4 [00:12<00:00,  3.08s/it, est. speed input: 3.32 toks/s, output: 83.04 toks/s]

Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3600.26it/s]

Processed prompts: 100%|██████████| 4/4 [00:12<00:00,  3.04s/it, est. speed input: 3.29 toks/s, output: 84.29 toks/s]


In [14]:
# Benchmark: Batched throughput with Adapter B
print("=" * 60)
print(f"BENCHMARK: Batched (bs={BATCH_SIZE}, adapter_b)")
print("=" * 60)

lora_req_b = LoRARequest("adapter_b", lora_int_id=1, lora_path=" ")

# Warmup
print(f"  Warmup (batch of {BATCH_SIZE})...")
warmup_inputs = [{"prompt": p} for p in batches[0]]
warmup_loras = [lora_req_b] * BATCH_SIZE
llm.generate(warmup_inputs, sampling_bench, lora_request=warmup_loras)

batch_results_b = []
print(f"  Running {n_batches} batches of {BATCH_SIZE}...")
for batch_idx, batch_prompts in enumerate(batches):
    batch_inputs = [{"prompt": p} for p in batch_prompts]
    batch_loras = [lora_req_b] * len(batch_prompts)
    
    t0 = time.time()
    outputs = llm.generate(batch_inputs, sampling_bench, lora_request=batch_loras)
    latency = time.time() - t0
    
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    aggregate_tok_s = total_tokens / latency if latency > 0 else 0
    per_request_tok_s = aggregate_tok_s / len(batch_prompts)
    
    batch_results_b.append({
        "batch_idx": batch_idx,
        "num_prompts": len(batch_prompts),
        "total_tokens": total_tokens,
        "latency_s": round(latency, 3),
        "aggregate_tok_s": round(aggregate_tok_s, 1),
        "per_request_tok_s": round(per_request_tok_s, 1),
    })
    print(f"    Batch {batch_idx}: {total_tokens} tokens, {latency:.2f}s, "
          f"{aggregate_tok_s:.1f} agg tok/s, {per_request_tok_s:.1f} per-req tok/s")

batch_agg_throughputs_b = [r["aggregate_tok_s"] for r in batch_results_b]
batch_per_req_throughputs_b = [r["per_request_tok_s"] for r in batch_results_b]
batch_latencies_b = [r["latency_s"] for r in batch_results_b]

print(f"\n  Batched results (bs={BATCH_SIZE}, adapter_b):")
print(f"    Mean aggregate throughput: {statistics.mean(batch_agg_throughputs_b):.1f} +/- {statistics.stdev(batch_agg_throughputs_b):.1f} tok/s")
print(f"    Mean per-request throughput: {statistics.mean(batch_per_req_throughputs_b):.1f} tok/s")
print(f"    Mean latency (wall clock per batch): {statistics.mean(batch_latencies_b):.2f}s")

BENCHMARK: Batched (bs=4, adapter_b)
  Warmup (batch of 4)...
  Running 3 batches of 4...
    Batch 0: 1024 tokens, 11.93s, 85.8 agg tok/s, 21.5 per-req tok/s
    Batch 1: 1024 tokens, 12.14s, 84.3 agg tok/s, 21.1 per-req tok/s
    Batch 2: 1024 tokens, 12.38s, 82.7 agg tok/s, 20.7 per-req tok/s

  Batched results (bs=4, adapter_b):
    Mean aggregate throughput: 84.3 +/- 1.6 tok/s
    Mean per-request throughput: 21.1 tok/s
    Mean latency (wall clock per batch): 12.15s



Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3930.93it/s]

Processed prompts: 100%|██████████| 4/4 [00:11<00:00,  2.99s/it, est. speed input: 3.34 toks/s, output: 85.58 toks/s]

Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3322.22it/s]

Processed prompts: 100%|██████████| 4/4 [00:11<00:00,  2.98s/it, est. speed input: 3.35 toks/s, output: 85.83 toks/s]

Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3799.19it/s]

Processed prompts: 100%|██████████| 4/4 [00:12<00:00,  3.04s/it, est. speed input: 3.38 toks/s, output: 84.33 toks/s]

Adding requests: 100%|██████████| 4/4 [00:00<00:00, 3604.90it/s]

Processed prompts: 100%|██████████| 4/4 [00:12<00:00,  3.09s/it, est. speed input: 3.23 toks/s, output: 82.75 toks/s]


In [15]:
# Summary comparison
single_mean = statistics.mean(single_throughputs)
batch_mean_a = statistics.mean(batch_agg_throughputs)
batch_mean_b = statistics.mean(batch_agg_throughputs_b)

print("\n" + "=" * 70)
print(f"BENCHMARK SUMMARY: Qwen3-14B on trn2.3xlarge, tp=4, BF16")
print("=" * 70)
print(f"{'Config':<35} {'Agg tok/s':>12} {'Per-req tok/s':>15} {'Latency':>10}")
print("-" * 70)
print(f"{'bs=1 single (adapter_a)':<35} {single_mean:>12.1f} {single_mean:>15.1f} {statistics.mean(single_latencies):>9.2f}s")
print(f"{'bs=4 batched (adapter_a)':<35} {batch_mean_a:>12.1f} {statistics.mean(batch_per_req_throughputs):>15.1f} {statistics.mean(batch_latencies):>9.2f}s")
print(f"{'bs=4 batched (adapter_b)':<35} {batch_mean_b:>12.1f} {statistics.mean(batch_per_req_throughputs_b):>15.1f} {statistics.mean(batch_latencies_b):>9.2f}s")
print("-" * 70)

speedup_a = batch_mean_a / single_mean if single_mean > 0 else 0
speedup_b = batch_mean_b / single_mean if single_mean > 0 else 0
print(f"\nSpeedup (bs=4 vs bs=1):")
print(f"  Adapter A: {speedup_a:.2f}x aggregate throughput")
print(f"  Adapter B: {speedup_b:.2f}x aggregate throughput")
print(f"\nbs=1 baseline from previous notebook: 55.3 tok/s")

print(f"\nConfiguration:")
print(f"  Instance: trn2.3xlarge (4 NeuronCores at LNC=2)")
print(f"  TP: {TP_DEGREE}, Compiled batch size: {BATCH_SIZE}")
print(f"  Max model len: {MAX_MODEL_LEN}")
print(f"  Precision: BF16, ISA kernels: QKV+Attn ON, MLP OFF")
print(f"  Compile time: {compile_time:.0f}s")


BENCHMARK SUMMARY: Qwen3-14B on trn2.3xlarge, tp=4, BF16
Config                                 Agg tok/s   Per-req tok/s    Latency
----------------------------------------------------------------------
bs=1 single (adapter_a)                     33.7            33.7      7.60s
bs=4 batched (adapter_a)                    83.5            20.9     12.27s
bs=4 batched (adapter_b)                    84.3            21.1     12.15s
----------------------------------------------------------------------

Speedup (bs=4 vs bs=1):
  Adapter A: 2.48x aggregate throughput
  Adapter B: 2.50x aggregate throughput

bs=1 baseline from previous notebook: 55.3 tok/s

Configuration:
  Instance: trn2.3xlarge (4 NeuronCores at LNC=2)
  TP: 4, Compiled batch size: 4
  Max model len: 4096
  Precision: BF16, ISA kernels: QKV+Attn ON, MLP OFF
  Compile time: 1072s


In [16]:
# Save results
results_path = os.path.expanduser("~/qwen3_14b_lora_bs4_results.json")
all_results = {
    "config": {
        "model": BASE_MODEL_ID,
        "instance": "trn2.3xlarge",
        "tp_degree": TP_DEGREE,
        "compiled_batch_size": BATCH_SIZE,
        "max_model_len": MAX_MODEL_LEN,
        "precision": "bfloat16",
        "isa_kernels": "qkv+attn_on_mlp_off",
        "lnc": 2,
        "compile_time_s": compile_time,
    },
    "single_request": {
        "mean_throughput_tok_s": round(statistics.mean(single_throughputs), 1),
        "std_throughput_tok_s": round(statistics.stdev(single_throughputs), 1),
        "mean_latency_s": round(statistics.mean(single_latencies), 2),
        "results": single_results,
    },
    "batched_adapter_a": {
        "batch_size": BATCH_SIZE,
        "mean_aggregate_tok_s": round(statistics.mean(batch_agg_throughputs), 1),
        "std_aggregate_tok_s": round(statistics.stdev(batch_agg_throughputs), 1),
        "mean_per_request_tok_s": round(statistics.mean(batch_per_req_throughputs), 1),
        "mean_latency_s": round(statistics.mean(batch_latencies), 2),
        "speedup_vs_single": round(speedup_a, 2),
        "results": batch_results,
    },
    "batched_adapter_b": {
        "batch_size": BATCH_SIZE,
        "mean_aggregate_tok_s": round(statistics.mean(batch_agg_throughputs_b), 1),
        "std_aggregate_tok_s": round(statistics.stdev(batch_agg_throughputs_b), 1),
        "mean_per_request_tok_s": round(statistics.mean(batch_per_req_throughputs_b), 1),
        "mean_latency_s": round(statistics.mean(batch_latencies_b), 2),
        "speedup_vs_single": round(speedup_b, 2),
        "results": batch_results_b,
    },
}

with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to {results_path}")

Results saved to /home/ubuntu/qwen3_14b_lora_bs4_results.json


In [17]:
print("Done! Batch size testing complete.")
print(f"\nKey finding: bs={BATCH_SIZE} aggregate throughput is {speedup_a:.2f}x single-request.")
print(f"Compile time: {compile_time:.0f}s ({compile_time/60:.1f} min)")

Done! Batch size testing complete.

Key finding: bs=4 aggregate throughput is 2.48x single-request.
Compile time: 1072s (17.9 min)
